# LeetCode 1486 — XOR Operation in an Array bằng Python

**Link đề:** https://leetcode.com/problems/xor-operation-in-an-array/
**Độ khó:** Easy · **Chủ đề:** Math, Bit Manipulation (thao tác bit)

```python
class Solution:
    def xorOperation(self, n: int, start: int) -> int:
        ...
```

### Mục tiêu học
1. Hiểu phép **XOR** (`^`) trên bit: bảng chân trị, cách tính tay trên số nhị phân.
2. Nắm các tính chất vàng của XOR: `a ^ a = 0`, `a ^ 0 = a`, giao hoán, kết hợp.
3. Viết lời giải mô phỏng: dựng mảng rồi XOR dồn (`O(n)`), và bản không cần mảng.
4. Hiểu lời giải toán học `O(1)`: công thức XOR từ `0` tới `x` theo chu kỳ 4, tách bit thấp nhất.
5. Trace, chứng minh, test đối chiếu và tự giải các biến thể XOR.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho số nguyên `n` và `start`. Định nghĩa mảng `nums` có độ dài `n` với:

```
nums[i] = start + 2 * i      (i = 0, 1, ..., n-1)
```

Trả về **XOR theo bit** (bitwise XOR) của tất cả phần tử trong `nums`.

| n | start | nums | Output |
|---:|---:|---|---:|
| 5 | 0 | `[0, 2, 4, 6, 8]` | `0^2^4^6^8 = 8` |
| 4 | 3 | `[3, 5, 7, 9]` | `3^5^7^9 = 8` |

**Constraints:** `1 <= n <= 1000`, `0 <= start <= 1000`.

Câu hỏi đặt ra:
- `^` trong Python là gì? (KHÔNG phải lũy thừa! Lũy thừa là `**`.)
- Có cần thật sự tạo mảng không?
- Có công thức tính ngay mà không cần vòng lặp?

In [1]:
n, start = 5, 0
nums = [start + 2 * i for i in range(n)]
print("nums =", nums)
n, start = 4, 3
print("nums =", [start + 2 * i for i in range(n)])
print("Chú ý: 2 ^ 3 =", 2 ^ 3, "còn 2 ** 3 =", 2 ** 3)

nums = [0, 2, 4, 6, 8]
nums = [3, 5, 7, 9]
Chú ý: 2 ^ 3 = 1 còn 2 ** 3 = 8


## 2. Nền tảng: hệ nhị phân và phép XOR

### 2.1. Nhị phân nhanh

Mỗi số nguyên không âm được viết bằng các bit 0/1; vị trí thứ `k` (đếm từ phải, từ 0) có giá trị `2^k`.
Ví dụ `6 = 110₂ = 4 + 2`. Hàm `bin(x)` cho chuỗi `'0b...'`, còn `format(x, '04b')` cho chuỗi 4 bit có đệm số 0.

In [2]:
for x in [0, 2, 3, 5, 6, 8, 9]:
    print(f"{x:>2} -> bin = {bin(x):>6} | 4 bit = {format(x, '04b')}")

 0 -> bin =    0b0 | 4 bit = 0000
 2 -> bin =   0b10 | 4 bit = 0010
 3 -> bin =   0b11 | 4 bit = 0011
 5 -> bin =  0b101 | 4 bit = 0101
 6 -> bin =  0b110 | 4 bit = 0110
 8 -> bin = 0b1000 | 4 bit = 1000
 9 -> bin = 0b1001 | 4 bit = 1001


### 2.2. XOR là gì?

XOR (exclusive OR — "hoặc loại trừ") so sánh **từng cặp bit cùng vị trí**: kết quả là `1` nếu hai bit **khác nhau**, `0` nếu **giống nhau**.

| a | b | a XOR b |
|:-:|:-:|:-:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Tính tay `3 ^ 5`:
```
  011   (3)
^ 101   (5)
-----
  110   (6)
```

In [3]:
def show_xor(a, b, width=4):
    print(f"  {format(a, f'0{width}b')}  ({a})")
    print(f"^ {format(b, f'0{width}b')}  ({b})")
    print(f"  {'-' * width}")
    print(f"  {format(a ^ b, f'0{width}b')}  ({a ^ b})")

show_xor(3, 5)
print()
show_xor(6, 7)

  0011  (3)
^ 0101  (5)
  ----
  0110  (6)

  0110  (6)
^ 0111  (7)
  ----
  0001  (1)


### 2.3. Bốn tính chất vàng của XOR

1. `a ^ 0 = a` — XOR với 0 không đổi gì (0 là "phần tử trung hòa").
2. `a ^ a = 0` — XOR một số với chính nó luôn ra 0 (tự triệt tiêu).
3. **Giao hoán:** `a ^ b = b ^ a`.
4. **Kết hợp:** `(a ^ b) ^ c = a ^ (b ^ c)` → thứ tự XOR không quan trọng.

Hệ quả: trong một dãy XOR, mọi số xuất hiện **chẵn** lần đều biến mất.

In [4]:
a, b_, c = 13, 7, 9
print("a ^ 0 =", a ^ 0)
print("a ^ a =", a ^ a)
print("a ^ b == b ^ a:", a ^ b_ == b_ ^ a)
print("(a^b)^c == a^(b^c):", (a ^ b_) ^ c == a ^ (b_ ^ c))
print("5 ^ 9 ^ 5 =", 5 ^ 9 ^ 5, "<- hai số 5 triệt tiêu")

a ^ 0 = 13
a ^ a = 0
a ^ b == b ^ a: True
(a^b)^c == a^(b^c): True
5 ^ 9 ^ 5 = 9 <- hai số 5 triệt tiêu


### 2.4. Toán tử `^=` và giá trị khởi đầu

`x ^= y` nghĩa là `x = x ^ y`. Khi XOR dồn một dãy, khởi tạo `result = 0` (vì `0 ^ a = a`) — giống như cộng dồn bắt đầu từ 0.

In [5]:
result = 0
for x in [3, 5, 7, 9]:
    result ^= x
    print(f"^ {x} -> result = {result} ({format(result, '04b')})")

^ 3 -> result = 3 (0011)
^ 5 -> result = 6 (0110)
^ 7 -> result = 1 (0001)
^ 9 -> result = 8 (1000)


## 3. Cách 1 — Mô phỏng: dựng mảng rồi XOR dồn

Đúng như gợi ý của đề: tạo `nums`, rồi XOR từng phần tử.

In [6]:
def xor_operation_array(n: int, start: int) -> int:
    nums = [start + 2 * i for i in range(n)]
    result = 0
    for x in nums:
        result ^= x
    return result

print(xor_operation_array(5, 0))   # 8
print(xor_operation_array(4, 3))   # 8

8
8


## 4. Cách 2 — Mô phỏng không cần mảng

Mảng chỉ dùng một lần rồi bỏ → không cần lưu. Tính `start + 2*i` ngay trong vòng lặp: bộ nhớ phụ `O(1)`.

In [7]:
def xor_operation_loop(n: int, start: int) -> int:
    result = 0
    for i in range(n):
        result ^= start + 2 * i     # + và * được tính TRƯỚC ^ (xem lỗi 2 ở mục 15)
    return result

print(xor_operation_loop(5, 0))
print(xor_operation_loop(4, 3))

8
8


### Code nộp LeetCode — mô phỏng (lời giải chính, dễ hiểu nhất)

In [8]:
class Solution:
    def xorOperation(self, n: int, start: int) -> int:
        result = 0
        for i in range(n):
            result ^= start + 2 * i
        return result

print(Solution().xorOperation(5, 0), Solution().xorOperation(4, 3))

8 8


## 5. TRACE từng bước — cách 2

Ví dụ `n = 4, start = 3`:

| i | nums[i] = 3 + 2i | nhị phân | result trước | result sau |
|---:|---:|---|---|---|
| 0 | 3 | `0011` | `0000` (0) | `0011` (3) |
| 1 | 5 | `0101` | `0011` (3) | `0110` (6) |
| 2 | 7 | `0111` | `0110` (6) | `0001` (1) |
| 3 | 9 | `1001` | `0001` (1) | `1000` (8) |

Kết quả `8` ✔.

In [9]:
def trace_xor_operation(n, start):
    result = 0
    for i in range(n):
        x = start + 2 * i
        before = result
        result ^= x
        print(f"i={i}: x={x:>2} ({format(x, '04b')}) | {format(before, '04b')} ^ {format(x, '04b')} = {format(result, '04b')} ({result})")
    print("Kết quả:", result)
    return result

trace_xor_operation(4, 3)
print()
trace_xor_operation(5, 0)

i=0: x= 3 (0011) | 0000 ^ 0011 = 0011 (3)
i=1: x= 5 (0101) | 0011 ^ 0101 = 0110 (6)
i=2: x= 7 (0111) | 0110 ^ 0111 = 0001 (1)
i=3: x= 9 (1001) | 0001 ^ 1001 = 1000 (8)
Kết quả: 8

i=0: x= 0 (0000) | 0000 ^ 0000 = 0000 (0)
i=1: x= 2 (0010) | 0000 ^ 0010 = 0010 (2)
i=2: x= 4 (0100) | 0010 ^ 0100 = 0110 (6)
i=3: x= 6 (0110) | 0110 ^ 0110 = 0000 (0)
i=4: x= 8 (1000) | 0000 ^ 1000 = 1000 (8)
Kết quả: 8


8

## 6. Cách 3 — `functools.reduce` với `operator.xor`

`reduce(f, dãy, khởi_tạo)` áp dụng `f` dồn từ trái sang phải: `f(f(f(khởi_tạo, a0), a1), a2)...`. `operator.xor(a, b)` chính là `a ^ b` dưới dạng hàm.

In [10]:
from functools import reduce
import operator

print(operator.xor(3, 5))                                          # 6
print(reduce(operator.xor, [3, 5, 7, 9], 0))                       # 8
print(reduce(operator.xor, range(0, 2 * 5, 2), 0))                 # n=5, start=0 -> 8

6
8
8


`range(start, start + 2*n, 2)` sinh đúng dãy `start, start+2, ..., start+2(n-1)` — bước nhảy 2 thay cho công thức `start + 2*i`.

In [11]:
def xor_operation_reduce(n: int, start: int) -> int:
    return reduce(operator.xor, range(start, start + 2 * n, 2), 0)

print(list(range(3, 3 + 2 * 4, 2)), "->", xor_operation_reduce(4, 3))

[3, 5, 7, 9] -> 8


## 7. Nút thắt — có thể nhanh hơn `O(n)` không?

Với `n <= 1000` thì `O(n)` là quá đủ. Nhưng nếu `n` tới `10^18` (câu hỏi phỏng vấn mở rộng), vòng lặp không chạy nổi. Ta cần **công thức đóng** `O(1)`.

Ý tưởng gồm 2 bước:
1. Biết công thức nhanh cho `0 ^ 1 ^ 2 ^ ... ^ x` (XOR các số **liên tiếp**).
2. Biến dãy "cách nhau 2" thành dãy liên tiếp bằng cách **tách bit thấp nhất**.

## 8. Insight 1 — XOR từ 0 tới x có chu kỳ 4

Gọi `f(x) = 0 ^ 1 ^ 2 ^ ... ^ x`. Cùng in ra để tìm quy luật:

In [12]:
acc = 0
for x in range(17):
    acc ^= x
    print(f"x={x:>2} (x % 4 = {x % 4}) -> f(x) = {acc:>2}")

x= 0 (x % 4 = 0) -> f(x) =  0
x= 1 (x % 4 = 1) -> f(x) =  1
x= 2 (x % 4 = 2) -> f(x) =  3
x= 3 (x % 4 = 3) -> f(x) =  0
x= 4 (x % 4 = 0) -> f(x) =  4
x= 5 (x % 4 = 1) -> f(x) =  1
x= 6 (x % 4 = 2) -> f(x) =  7
x= 7 (x % 4 = 3) -> f(x) =  0
x= 8 (x % 4 = 0) -> f(x) =  8
x= 9 (x % 4 = 1) -> f(x) =  1
x=10 (x % 4 = 2) -> f(x) = 11
x=11 (x % 4 = 3) -> f(x) =  0
x=12 (x % 4 = 0) -> f(x) = 12
x=13 (x % 4 = 1) -> f(x) =  1
x=14 (x % 4 = 2) -> f(x) = 15
x=15 (x % 4 = 3) -> f(x) =  0
x=16 (x % 4 = 0) -> f(x) = 16


Quy luật:

| `x % 4` | `f(x)` |
|:-:|:-:|
| 0 | `x` |
| 1 | `1` |
| 2 | `x + 1` |
| 3 | `0` |

**Vì sao?** Với mọi số chẵn `k`, cặp `k ^ (k+1) = 1` (chỉ khác nhau ở bit cuối). Nhóm `0^1^2^3 = 1^1 = 0`, nên cứ mỗi khối 4 số bắt đầu từ bội của 4 thì XOR = 0. Vậy `f(x)` chỉ phụ thuộc vào vài số cuối:
- `x % 4 == 3`: các khối 4 trọn vẹn → `0`.
- `x % 4 == 0`: phần dư chỉ có `x` → `x`.
- `x % 4 == 1`: phần dư `x-1 ^ x` = 1 (cặp chẵn-lẻ) → `1`.
- `x % 4 == 2`: phần dư `(x-2 ^ x-1) ^ x = 1 ^ x = x + 1` (x chẵn nên bit cuối là 0, XOR 1 = cộng 1).

In [13]:
def xor_0_to(x: int) -> int:
    # f(x) = 0 ^ 1 ^ ... ^ x ; quy ước f(-1) = 0 (dãy rỗng)
    if x < 0:
        return 0
    return [x, 1, x + 1, 0][x % 4]

acc = 0
for x in range(200):
    acc ^= x
    assert xor_0_to(x) == acc
print("Công thức f(x) đúng với x = 0..199")
print("k ^ (k+1) với k chẵn:", [k ^ (k + 1) for k in range(0, 10, 2)])

Công thức f(x) đúng với x = 0..199
k ^ (k+1) với k chẵn: [1, 1, 1, 1, 1]


Có `f`, XOR đoạn liên tiếp `a ^ (a+1) ^ ... ^ b` = `f(b) ^ f(a-1)` — vì `f(a-1)` xuất hiện trong `f(b)` và XOR lần nữa thì triệt tiêu (tính chất `t ^ t = 0`). Giống hệt "tổng đoạn = prefix[b] - prefix[a-1]", chỉ thay `-` bằng `^`.

In [14]:
a, b2 = 5, 9
direct = 0
for v in range(a, b2 + 1):
    direct ^= v
print("trực tiếp:", direct, "| công thức f(b) ^ f(a-1):", xor_0_to(b2) ^ xor_0_to(a - 1))

trực tiếp: 5 | công thức f(b) ^ f(a-1): 5


## 9. Insight 2 — tách bit thấp nhất để biến "cách 2" thành "liên tiếp"

Mọi `nums[i] = start + 2i` có **cùng tính chẵn lẻ** với `start` → bit thấp nhất (bit 0) của mọi phần tử đều bằng `b0 = start & 1`.

Viết `nums[i] = 2 * (s + i) + b0` với `s = start >> 1` (`>> 1` là dịch phải 1 bit = chia nguyên cho 2).

Khi XOR, các bit được xử lý **độc lập**:
- **Phần bit cao:** `(s+0), (s+1), ..., (s+n-1)` là dãy **liên tiếp** → XOR = `f(s+n-1) ^ f(s-1)`, rồi dịch trái 1 bit (`<< 1`, nhân 2) để về đúng vị trí.
- **Bit thấp:** XOR của `n` bản sao bit `b0` = `b0` nếu `n` lẻ, `0` nếu `n` chẵn. Viết gọn: `n & start & 1` (bằng 1 chỉ khi cả `n` lẻ **và** `start` lẻ).

```
answer = (f(s + n - 1) ^ f(s - 1)) << 1  |  (n & start & 1)
```

`|` là OR theo bit; ở đây bit 0 của phần trái luôn là 0 nên `|` tương đương "ghép" bit thấp vào.

In [15]:
start = 7
print("start =", start, "=", format(start, '04b'))
print("start & 1  =", start & 1, " (bit thấp nhất)")
print("start >> 1 =", start >> 1, " (= start // 2)")
print("(start >> 1) << 1 | (start & 1) =", ((start >> 1) << 1) | (start & 1), "<- ghép lại đúng start")
n = 4
print("nums       =", [start + 2 * i for i in range(n)])
print("phần cao   =", [(start >> 1) + i for i in range(n)], "<- liên tiếp!")

start = 7 = 0111
start & 1  = 1  (bit thấp nhất)
start >> 1 = 3  (= start // 2)
(start >> 1) << 1 | (start & 1) = 7 <- ghép lại đúng start
nums       = [7, 9, 11, 13]
phần cao   = [3, 4, 5, 6] <- liên tiếp!


## 10. Cách 4 — Lời giải toán học `O(1)`

In [16]:
def xor_operation_math(n: int, start: int) -> int:
    s = start >> 1                                  # phần bit cao của start
    high = xor_0_to(s + n - 1) ^ xor_0_to(s - 1)    # XOR dãy liên tiếp s..s+n-1
    low = n & start & 1                             # bit 0: =1 chỉ khi start lẻ và n lẻ
    return (high << 1) | low

print(xor_operation_math(5, 0))   # 8
print(xor_operation_math(4, 3))   # 8
print(xor_operation_math(10**18 + 1, 12345))   # chạy ngay lập tức dù n khổng lồ

8
8
2000000000000012345


### Trace cách 4 với `n = 4, start = 3`

| Bước | Giá trị |
|---|---|
| `s = 3 >> 1` | `1` |
| dãy cao `s..s+n-1` | `1, 2, 3, 4` |
| `f(4) ^ f(0)` | `4 ^ 0 = 4` (kiểm tra: `1^2^3^4 = 4`) |
| `high << 1` | `8` |
| `low = 4 & 3 & 1` | `0` (n chẵn) |
| kết quả | `8 | 0 = 8` ✔ |

In [17]:
def trace_math(n, start):
    s = start >> 1
    print(f"s = {start} >> 1 = {s}; dãy cao = {list(range(s, s + n))}")
    fa, fb = xor_0_to(s - 1), xor_0_to(s + n - 1)
    high = fb ^ fa
    print(f"f({s + n - 1}) = {fb}, f({s - 1}) = {fa}, high = {high}")
    low = n & start & 1
    print(f"low = {n} & {start} & 1 = {low}")
    ans = (high << 1) | low
    print(f"answer = ({high} << 1) | {low} = {ans}")
    return ans

trace_math(4, 3)
print()
trace_math(3, 5)    # nums = [5, 7, 9] -> 5^7^9 = 11

s = 3 >> 1 = 1; dãy cao = [1, 2, 3, 4]
f(4) = 4, f(0) = 0, high = 4
low = 4 & 3 & 1 = 0
answer = (4 << 1) | 0 = 8

s = 5 >> 1 = 2; dãy cao = [2, 3, 4]
f(4) = 4, f(1) = 1, high = 5
low = 3 & 5 & 1 = 1
answer = (5 << 1) | 1 = 11


11

## 11. Vì sao đúng?

**Cách 2 (mô phỏng):** invariant — sau vòng `i`, `result = nums[0] ^ ... ^ nums[i]`. Khởi đầu `result = 0` = XOR của dãy rỗng; mỗi vòng `result ^= nums[i]` mở rộng thêm một phần tử. Hết vòng → XOR toàn bộ.

**Cách 4 (toán học):** XOR hoạt động **độc lập trên từng bit** (bit `k` của kết quả chỉ phụ thuộc bit `k` của các toán hạng). Viết `nums[i] = 2(s+i) + b0`:
- Bit 0 của mọi phần tử là `b0` → XOR của `n` bit giống nhau = `b0` nếu `n` lẻ, ngược lại 0.
- Các bit từ 1 trở lên của `nums[i]` chính là các bit của `s+i` → XOR chúng = XOR dãy liên tiếp `s..s+n-1` = `f(s+n-1) ^ f(s-1)` (do `f(s-1)` triệt tiêu phần đầu).

Ghép hai phần lại được đúng đáp án.

## 12. Code nộp LeetCode — cách `O(1)`

In [18]:
class SolutionMath:
    def xorOperation(self, n: int, start: int) -> int:
        def f(x: int) -> int:          # 0 ^ 1 ^ ... ^ x, f(-1) = 0
            if x < 0:
                return 0
            return [x, 1, x + 1, 0][x % 4]
        s = start >> 1
        return ((f(s + n - 1) ^ f(s - 1)) << 1) | (n & start & 1)

assert SolutionMath().xorOperation(5, 0) == 8
assert SolutionMath().xorOperation(4, 3) == 8
print("SolutionMath PASS ví dụ đề")

SolutionMath PASS ví dụ đề


(Trên LeetCode, đổi tên `SolutionMath` thành `Solution` khi nộp.)

## 13. Phân tích Time / Space

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Dựng mảng + XOR | `O(n)` | `O(n)` | sát đề nhất |
| 2. Vòng lặp không mảng | `O(n)` | `O(1)` | nên dùng khi phỏng vấn cơ bản |
| 3. `reduce` | `O(n)` | `O(1)` | `range` sinh dần, không tạo list |
| 4. Công thức | `O(1)` | `O(1)` | cần hiểu tính chất XOR |

(Với số nguyên cỡ thông thường, mỗi phép `^`, `+`, `>>` coi là `O(1)`.)

## 14. Kiểm thử: mẫu đề, biên, đối chiếu

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<26} -> {actual}")

tests = [
    ("đề 1",                   5, 0, 8),
    ("đề 2",                   4, 3, 8),
    ("n=1 -> chính start",     1, 7, 7),
    ("n=1, start=0",           1, 0, 0),
    ("start lẻ, n lẻ",         3, 5, 5 ^ 7 ^ 9),
    ("start chẵn, n=2",        2, 10, 10 ^ 12),
    ("giới hạn trên",          1000, 1000, xor_operation_array(1000, 1000)),
]
for name, n, start, exp in tests:
    for f in (xor_operation_array, xor_operation_loop, xor_operation_reduce, xor_operation_math):
        assert f(n, start) == exp, (f.__name__, n, start)
    check(name, Solution().xorOperation(n, start), exp)

PASS  đề 1                       -> 8
PASS  đề 2                       -> 8
PASS  n=1 -> chính start         -> 7
PASS  n=1, start=0               -> 0
PASS  start lẻ, n lẻ             -> 11
PASS  start chẵn, n=2            -> 6
PASS  giới hạn trên              -> 0


In [20]:
# Đối chiếu toàn bộ miền constraints nhỏ: mọi n trong 1..60, start trong 0..200
for n in range(1, 61):
    for start in range(0, 201):
        a = xor_operation_loop(n, start)
        assert xor_operation_math(n, start) == a
        assert SolutionMath().xorOperation(n, start) == a
print("Cách 4 khớp cách 2 trên", 60 * 201, "cặp (n, start). PASS")

Cách 4 khớp cách 2 trên 12060 cặp (n, start). PASS


## 15. Lỗi người mới hay gặp

### Lỗi 1: Nhầm `^` với lũy thừa
```python
result = start ** 2   # nghĩ là XOR? SAI — đây là lũy thừa
```
Ngược lại, viết `2 ^ 3` để tính 2³ sẽ ra `1` chứ không phải `8`.

### Lỗi 2: Thứ tự ưu tiên toán tử
Trong Python, thứ tự ưu tiên (cao → thấp) là: `*` → `+ -` → `<< >>` → `&` → `^` → `|` → so sánh (`==`, `<`, …).
- Nhờ vậy `result ^= start + 2 * i` tính `start + 2*i` trước rồi mới XOR — đúng ý.
- Nhưng `1 << 1 + 1` lại là `1 << 2 = 4`, không phải `(1 << 1) + 1 = 3`.
- `3 + 2 ^ 1` là `(3 + 2) ^ 1 = 4`, không phải `3 + (2 ^ 1) = 6`.

Khi trộn phép bit với phép số học, **luôn thêm ngoặc** cho rõ.

### Lỗi 3: Khởi tạo `result = 1` (hoặc `result = start` rồi vẫn duyệt từ `i = 0`)
`start` bị XOR hai lần → triệt tiêu mất.

### Lỗi 4: Công thức `f(x)` quên `f(-1) = 0`
Khi `start` là 0 hoặc 1 thì `s = 0`, cần `f(s-1) = f(-1)`. Trong Python `-1 % 4 == 3` nên tra bảng ra `0` — "may mắn" đúng; nhưng ở C++/Java `-1 % 4 == -1` sẽ lỗi. Nên xử lý rõ ràng `x < 0`.

In [21]:
print("2 ^ 3  =", 2 ^ 3, "| 2 ** 3 =", 2 ** 3)
print("1 << 1 + 1   =", 1 << 1 + 1, "(là 1 << 2)")
print("(1 << 1) + 1 =", (1 << 1) + 1)
print("3 + 2 ^ 1    =", 3 + 2 ^ 1, "(là (3+2) ^ 1 = 5 ^ 1)")

def wrong_init_start(n, start):
    result = start                 # lỗi 3
    for i in range(n):
        result ^= start + 2 * i
    return result
print("lỗi 3:", wrong_init_start(4, 3), "| đúng:", xor_operation_loop(4, 3))
print("Python: -1 % 4 =", -1 % 4, "-> bảng tra [x,1,x+1,0][3] = 0 (đúng nhờ may mắn)")

2 ^ 3  = 1 | 2 ** 3 = 8
1 << 1 + 1   = 4 (là 1 << 2)
(1 << 1) + 1 = 3
3 + 2 ^ 1    = 4 (là (3+2) ^ 1 = 5 ^ 1)
lỗi 3: 11 | đúng: 8
Python: -1 % 4 = 3 -> bảng tra [x,1,x+1,0][3] = 0 (đúng nhờ may mắn)


### Mẹo nhận dạng pattern

- Thấy "XOR của nhiều số" → nghĩ tới tính chất **tự triệt tiêu** `a ^ a = 0` và **thứ tự không quan trọng**.
- Thấy "XOR một đoạn liên tiếp `a..b`" → `f(b) ^ f(a-1)` với `f` chu kỳ 4 (giống prefix sum).
- Thấy "dãy cách đều 2 / cùng tính chẵn lẻ" → **tách bit thấp nhất** ra xử lý riêng.
- Thấy "tìm số xuất hiện lẻ lần / số bị thiếu" → XOR toàn bộ (LeetCode 136 Single Number, 268 Missing Number).

### Biến thể

### Biến thể A — Single Number (LeetCode 136)
Mọi số xuất hiện 2 lần trừ một số. XOR tất cả → còn lại số đó.

In [22]:
def single_number(nums):
    res = 0
    for x in nums:
        res ^= x
    return res

print(single_number([4, 1, 2, 1, 2]))   # 4

4


### Biến thể B — Missing Number (LeetCode 268)
Mảng chứa `n` số phân biệt trong `[0, n]`, thiếu đúng một số. XOR mảng với `f(n)`: mọi số có mặt xuất hiện 2 lần và triệt tiêu.

In [23]:
def missing_number(nums):
    res = xor_0_to(len(nums))
    for x in nums:
        res ^= x
    return res

print(missing_number([3, 0, 1]))   # 2

2


## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (tính tay)
Tính tay `6 ^ 11` bằng cách viết 4 bit của mỗi số. Trả về kết quả.

In [24]:
def exercise_xor_by_hand() -> int:
    # TODO: viết 6 và 11 ở dạng 4 bit, XOR từng cột, đổi về thập phân
    pass

# assert exercise_xor_by_hand() == 6 ^ 11

### Bài 2 (đoán output)
Không chạy, đoán `xor_operation_loop(3, 1)` (nums = `[1, 3, 5]`).

In [25]:
def exercise_predict() -> int:
    # TODO
    pass

# assert exercise_predict() == xor_operation_loop(3, 1)

### Bài 3 (điền chỗ trống)

In [26]:
def exercise_xor_operation(n: int, start: int) -> int:
    # TODO 1: result = ?
    # TODO 2: for i in range(n): result ^= ?
    pass

# assert exercise_xor_operation(5, 0) == 8
# assert exercise_xor_operation(4, 3) == 8

### Bài 4 (XOR đoạn liên tiếp)
Viết `exercise_xor_range(a, b)` trả về `a ^ (a+1) ^ ... ^ b` trong `O(1)` (giả sử `0 <= a <= b`).

In [27]:
def exercise_xor_range(a: int, b: int) -> int:
    # TODO: dùng f(x) = 0 ^ ... ^ x
    pass

# assert exercise_xor_range(5, 9) == 5 ^ 6 ^ 7 ^ 8 ^ 9
# assert exercise_xor_range(0, 0) == 0

### Bài 5 (biến thể bước nhảy)
Cho `n, start, step` với `step` là **lũy thừa của 2** (1, 2, 4, …). Trả về XOR của `start + step*i` với `i = 0..n-1`. Viết bản mô phỏng `O(n)` là đủ.

In [28]:
def exercise_xor_with_step(n: int, start: int, step: int) -> int:
    # TODO
    pass

# assert exercise_xor_with_step(5, 0, 2) == 8
# assert exercise_xor_with_step(3, 1, 4) == 1 ^ 5 ^ 9

### Bài 6 (mini challenge — LeetCode 1720 Decode XORed Array)
Cho `encoded` với `encoded[i] = arr[i] ^ arr[i+1]` và `first = arr[0]`. Khôi phục `arr`.

Ví dụ `encoded = [1, 2, 3], first = 1` → `[1, 0, 2, 1]`.

In [29]:
def exercise_decode(encoded: list[int], first: int) -> list[int]:
    # TODO: arr[i+1] = arr[i] ^ encoded[i]  (vì sao?)
    pass

# assert exercise_decode([1, 2, 3], 1) == [1, 0, 2, 1]
# assert exercise_decode([6, 2, 7, 3], 4) == [4, 2, 0, 7, 4]

### Gợi ý ba mức

**Bài 1** — M1: XOR = "khác nhau thì 1". M2: `6 = 0110`, `11 = 1011`. M3: XOR từng cột → `1101`.

**Bài 2** — M1: lập bảng như mục 5. M2: `1 ^ 3 = 2`. M3: `2 ^ 5 = 7`.

**Bài 3** — M1: XOR dồn bắt đầu từ phần tử trung hòa. M2: `result = 0`. M3: `result ^= start + 2 * i`.

**Bài 4** — M1: giống tổng đoạn bằng prefix. M2: `f(b) ^ f(a - 1)`. M3: dùng `xor_0_to` đã định nghĩa, nhớ `f(-1) = 0`.

**Bài 5** — M1: chỉ đổi `2` thành `step`. M2: `range(start, start + step * n, step)`. M3: `result ^= start + step * i`.

**Bài 6** — M1: `encoded[i] ^ arr[i] = arr[i] ^ arr[i+1] ^ arr[i]`. M2: `arr[i]` xuất hiện hai lần nên triệt tiêu. M3: `arr = [first]`, rồi `arr.append(arr[-1] ^ e)` với mỗi `e`.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [30]:
def answer_xor_by_hand():
    return 0b1101           # 13

def answer_predict():
    return 7

def answer_xor_operation(n, start):
    result = 0
    for i in range(n):
        result ^= start + 2 * i
    return result

def answer_xor_range(a, b):
    return xor_0_to(b) ^ xor_0_to(a - 1)

def answer_xor_with_step(n, start, step):
    result = 0
    for i in range(n):
        result ^= start + step * i
    return result

def answer_decode(encoded, first):
    arr = [first]
    for e in encoded:
        arr.append(arr[-1] ^ e)
    return arr

assert answer_xor_by_hand() == 6 ^ 11
assert answer_predict() == xor_operation_loop(3, 1)
assert answer_xor_operation(5, 0) == 8 and answer_xor_operation(4, 3) == 8
assert answer_xor_range(5, 9) == 5 ^ 6 ^ 7 ^ 8 ^ 9
assert answer_xor_range(0, 0) == 0
for a in range(0, 40):
    for b2 in range(a, 40):
        assert answer_xor_range(a, b2) == reduce(operator.xor, range(a, b2 + 1), 0)
assert answer_xor_with_step(5, 0, 2) == 8
assert answer_xor_with_step(3, 1, 4) == 1 ^ 5 ^ 9
assert answer_decode([1, 2, 3], 1) == [1, 0, 2, 1]
assert answer_decode([6, 2, 7, 3], 4) == [4, 2, 0, 7, 4]
print("0b1101 =", 0b1101, "(tiền tố 0b viết số nhị phân trực tiếp)")
print("Tất cả đáp án PASS!")

0b1101 = 13 (tiền tố 0b viết số nhị phân trực tiếp)
Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def xorOperation(self, n: int, start: int) -> int:
        result = 0                     # 0 là phần tử trung hòa của XOR
        for i in range(n):
            result ^= start + 2 * i
        return result
```

Công thức nâng cao:
```python
f = lambda x: 0 if x < 0 else [x, 1, x + 1, 0][x % 4]   # 0 ^ 1 ^ ... ^ x
s = start >> 1
answer = ((f(s + n - 1) ^ f(s - 1)) << 1) | (n & start & 1)
```

**Checklist trước khi submit:**
- [ ] Dùng `^` (XOR), không phải `**`?
- [ ] `result` khởi tạo bằng `0`?
- [ ] Có ngoặc rõ ràng khi trộn `<<`, `|`, `+`?
- [ ] Nếu dùng công thức: xử lý `f(-1) = 0`, bit thấp `n & start & 1`?
- [ ] Đã test `n = 1` và cả `start` chẵn/lẻ?